# 3. Data Analysis with SQL
## Đề tài: Phân tích và Dự đoán Phí Vận chuyển (Freight) trên Sàn Thương mại điện tử



**Nguồn dữ liệu**: `olist_freight_clean.csv` — 109.473 dòng, cấp độ 1 sản phẩm/1 đơn hàng, đã qua đầy đủ các bước làm sạch (loại trùng lặp, loại giá trị thiếu/không hợp lệ, kiểm tra tính nhất quán).

## Bước 0: Nạp dữ liệu vào SQLite

In [29]:
from google.colab import drive
import pandas as pd
import sqlite3

# 1. Kết nối Google Drive
drive.mount('/content/drive')

# 2. Đường dẫn file dữ liệu đã làm sạch
file_path = '/content/drive/MyDrive/ADY201M/data/clean/olist_freight_clean (2).csv'

# 3. Đọc dữ liệu và tạo bảng SQLite trong bộ nhớ
df = pd.read_csv(file_path)
conn = sqlite3.connect(':memory:')
df.to_sql('freight_data', conn, index=False, if_exists='replace')

print(f'Bảng "freight_data" đã tạo thành công với {len(df):,} bản ghi!')
print(f'Các cột: {list(df.columns)}\n')

def sql(query):
    return pd.read_sql_query(query, conn)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Bảng "freight_data" đã tạo thành công với 109,473 bản ghi!
Các cột: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'month', 't_days', 'freight_value', 'price', 'category', 'weight_g', 'length_cm', 'height_cm', 'width_cm', 'volume_cm3', 'seller_state', 'customer_state', 'same_state', 'same_zip', 'seller_zip', 'customer_zip', 'seller_lat', 'seller_lng', 'customer_lat', 'customer_lng', 'distance_km', 'order_n_items', 'geo_imputed', 'category_imputed', 'freight_shared_flag']



### Kiểm tra cấu trúc bảng

In [30]:
print("=== KIỂM TRA CẤU TRÚC BẢNG ===")
print(sql("PRAGMA table_info(freight_data);"))


=== KIỂM TRA CẤU TRÚC BẢNG ===
    cid                      name     type  notnull dflt_value  pk
0     0                  order_id     TEXT        0       None   0
1     1             order_item_id  INTEGER        0       None   0
2     2                product_id     TEXT        0       None   0
3     3                 seller_id     TEXT        0       None   0
4     4               customer_id     TEXT        0       None   0
5     5              order_status     TEXT        0       None   0
6     6  order_purchase_timestamp     TEXT        0       None   0
7     7                     month     TEXT        0       None   0
8     8                    t_days  INTEGER        0       None   0
9     9             freight_value     REAL        0       None   0
10   10                     price     REAL        0       None   0
11   11                  category     TEXT        0       None   0
12   12                  weight_g  INTEGER        0       None   0
13   13                 length_

---
## Query 1 — Phí ship trung bình theo nhóm khoảng cách

**Câu hỏi phân tích**: Khoảng cách giao hàng (người bán → người mua) ảnh hưởng thế nào đến phí vận chuyển?
**Mục đích**: Kiểm chứng giả thuyết "khoảng cách càng xa, phí ship càng cao" — một trong những biến đầu vào (X) chính của mô hình Regression.

In [31]:
print("=== 1. PHÍ SHIP THEO NHÓM KHOẢNG CÁCH ===")
print(sql("""
SELECT
    CASE
        WHEN distance_km < 200  THEN '1. Gần (<200km)'
        WHEN distance_km < 800  THEN '2. Trung bình (200-800km)'
        WHEN distance_km < 1500 THEN '3. Xa (800-1500km)'
        ELSE '4. Rất xa (>1500km)'
    END AS nhom_khoang_cach,
    COUNT(*) AS so_don,
    ROUND(AVG(freight_value), 2) AS phi_ship_TB,
    ROUND(AVG(distance_km), 1) AS khoang_cach_TB_km
FROM freight_data
GROUP BY nhom_khoang_cach
ORDER BY nhom_khoang_cach;
"""))


=== 1. PHÍ SHIP THEO NHÓM KHOẢNG CÁCH ===
            nhom_khoang_cach  so_don  phi_ship_TB  khoang_cach_TB_km
0            1. Gần (<200km)   28155        12.67               68.2
1  2. Trung bình (200-800km)   54266        19.80              459.5
2         3. Xa (800-1500km)   17297        23.72             1030.7
3        4. Rất xa (>1500km)    9755        35.92             2110.9


**Nhận xét**: Phí ship tăng đều đặn theo khoảng cách — đơn hàng "rất xa" (>1500km) có phí trung bình cao gấp **~2,8 lần** so với đơn hàng "gần" (<200km). Xác nhận khoảng cách là yếu tố ảnh hưởng mạnh.

---
## Query 2 — So sánh phí ship: cùng bang vs khác bang

**Câu hỏi phân tích**: Giao hàng trong cùng 1 bang có rẻ hơn đáng kể so với giao khác bang không?
**Mục đích**: Đánh giá vai trò của biến phân loại địa lý (`same_state`) bên cạnh biến khoảng cách liên tục.

In [32]:
print("=== 2. SO SÁNH PHÍ SHIP: CÙNG BANG VS KHÁC BANG ===")
print(sql("""
SELECT
    CASE WHEN same_state = 1 THEN 'Cùng bang' ELSE 'Khác bang' END AS nhom,
    COUNT(*) AS so_don,
    ROUND(AVG(freight_value), 2) AS phi_ship_TB,
    ROUND(AVG(distance_km), 1) AS khoang_cach_TB
FROM freight_data
GROUP BY same_state;
"""))


=== 2. SO SÁNH PHÍ SHIP: CÙNG BANG VS KHÁC BANG ===
        nhom  so_don  phi_ship_TB  khoang_cach_TB
0  Khác bang   69797        23.74           847.8
1  Cùng bang   39676        13.48           153.8


**Nhận xét**: Đơn hàng khác bang có phí ship trung bình cao hơn **76%** so với cùng bang (23,74 R$ so với 13,48 R$), phần lớn do khoảng cách xa hơn (847,8km so với 153,8km).

---
## Query 3 — Top 10 danh mục sản phẩm có phí ship trung bình cao nhất

**Câu hỏi phân tích**: Danh mục sản phẩm nào có chi phí vận chuyển cao nhất, và có liên hệ gì với trọng lượng?
**Mục đích**: Kiểm tra xem phí ship cao có thực sự gắn liền với các mặt hàng nặng/cồng kềnh hay không (chỉ xét danh mục có ≥30 đơn để đảm bảo độ tin cậy thống kê).

In [33]:
print("=== 3. TOP 10 DANH MỤC PHÍ SHIP CAO NHẤT ===")
print(sql("""
SELECT
    category,
    COUNT(*) AS so_don,
    ROUND(AVG(freight_value), 2) AS phi_ship_TB,
    ROUND(AVG(weight_g), 0) AS trong_luong_TB
FROM freight_data
GROUP BY category
HAVING COUNT(*) >= 30
ORDER BY phi_ship_TB DESC
LIMIT 10;
"""))


=== 3. TOP 10 DANH MỤC PHÍ SHIP CAO NHẤT ===
                                  category  so_don  phi_ship_TB  \
0                                computers     199        48.57   
1                        home_appliances_2     231        44.39   
2        furniture_mattress_and_upholstery      37        42.74   
3  kitchen_dining_laundry_garden_furniture     274        42.12   
4                        furniture_bedroom     103        42.01   
5                         office_furniture    1661        40.20   
6    small_appliances_home_oven_and_coffee      73        36.39   
7                    furniture_living_room     495        35.75   
8                   signaling_and_security     197        32.88   
9           industry_commerce_and_business     261        28.76   

   trong_luong_TB  
0          7203.0  
1          8926.0  
2          7472.0  
3          8938.0  
4         10167.0  
5         11325.0  
6          5011.0  
7          8101.0  
8          3916.0  
9          6670.0

**Nhận xét**: Toàn bộ top 10 đều là nhóm nội thất/thiết bị gia dụng/máy tính — trọng lượng trung bình rất cao (5–11kg), nhất quán với giả thuyết trọng lượng là yếu tố ảnh hưởng mạnh.

---
## Query 4 — Phí ship trung bình theo nhóm trọng lượng

**Câu hỏi phân tích**: Trọng lượng sản phẩm ảnh hưởng định lượng như thế nào đến phí ship?
**Mục đích**: Đo trực tiếp mối quan hệ trọng lượng–phí ship bằng cách chia nhóm, bổ sung bằng chứng SQL độc lập cho bước Regression Analysis.

In [34]:
print("=== 4. PHÍ SHIP THEO NHÓM TRỌNG LƯỢNG ===")
print(sql("""
SELECT
    CASE
        WHEN weight_g < 500  THEN '1. Rất nhẹ (<500g)'
        WHEN weight_g < 2000 THEN '2. Nhẹ (500-2000g)'
        WHEN weight_g < 5000 THEN '3. Trung bình (2-5kg)'
        ELSE '4. Nặng (>5kg)'
    END AS nhom_trong_luong,
    COUNT(*) AS so_don,
    ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM freight_data
GROUP BY nhom_trong_luong
ORDER BY nhom_trong_luong;
"""))


=== 4. PHÍ SHIP THEO NHÓM TRỌNG LƯỢNG ===
        nhom_trong_luong  so_don  phi_ship_TB
0     1. Rất nhẹ (<500g)   43698        15.18
1     2. Nhẹ (500-2000g)   40851        17.89
2  3. Trung bình (2-5kg)   11298        22.82
3         4. Nặng (>5kg)   13626        39.61


**Nhận xét**: Phí ship tăng từ 15,18 R$ (rất nhẹ) lên 39,61 R$ (nặng) — tăng **2,6 lần**, củng cố thêm cho kết quả tương quan đã tính ở bước Python Analysis (r ≈ 0,61).

---
## Query 5 — Top 10 bang người bán có phí ship trung bình cao nhất

**Câu hỏi phân tích**: Người bán ở bang nào đang chịu/thu phí ship cao nhất trung bình?
**Mục đích**: Phát hiện các khu vực địa lý có chi phí logistics cao, hữu ích cho gợi ý vận hành (chỉ xét bang có ≥20 đơn).

In [35]:
print("=== 5. TOP 10 BANG SELLER PHÍ SHIP CAO NHẤT ===")
print(sql("""
SELECT
    seller_state,
    COUNT(*) AS so_don,
    ROUND(AVG(freight_value), 2) AS phi_ship_TB,
    ROUND(AVG(distance_km), 1) AS khoang_cach_TB
FROM freight_data
GROUP BY seller_state
HAVING COUNT(*) >= 20
ORDER BY phi_ship_TB DESC
LIMIT 10;
"""))


=== 5. TOP 10 BANG SELLER PHÍ SHIP CAO NHẤT ===
  seller_state  so_don  phi_ship_TB  khoang_cach_TB
0           CE      90        46.74          1665.7
1           PB      37        39.16          1473.4
2           ES     363        32.75           805.6
3           MT     144        32.05          1366.5
4           BA     623        30.83          1160.9
5           MA     402        30.03          1943.5
6           PE     445        27.69          1580.0
7           SC    3987        26.07           737.7
8           RS    2162        25.99           992.7
9           MS      47        24.22          1051.5


**Nhận xét**: Các bang có phí ship cao nhất (CE, PB, ES...) đều có khoảng cách trung bình đến khách hàng rất lớn (>800km) — xác nhận lại vai trò của vị trí địa lý người bán.

---
## Query 6 — Phí ship trung bình theo tháng (xu hướng mùa vụ)

**Câu hỏi phân tích**: Phí vận chuyển có biến động theo thời gian/mùa vụ trong năm không?
**Mục đích**: Kiểm tra yếu tố thời gian — nếu có biến động mùa vụ rõ rệt (VD: cao điểm mua sắm cuối năm), có thể cân nhắc thêm biến tháng/mùa vào mô hình Regression.

In [36]:
print("=== 6. PHÍ SHIP TRUNG BÌNH THEO THÁNG (XU HƯỚNG MÙA VỤ) ===")
print(sql("""
SELECT
    month,
    COUNT(*) AS so_don,
    ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM freight_data
GROUP BY month
ORDER BY month;
"""))


=== 6. PHÍ SHIP TRUNG BÌNH THEO THÁNG (XU HƯỚNG MÙA VỤ) ===
      month  so_don  phi_ship_TB
0   2017-01     912        17.17
1   2017-02    1858        19.92
2   2017-03    2894        19.03
3   2017-04    2562        19.53
4   2017-05    4004        19.36
5   2017-06    3486        19.53
6   2017-07    4414        19.18
7   2017-08    4785        19.04
8   2017-09    4736        19.78
9   2017-10    5214        19.73
10  2017-11    8475        19.54
11  2017-12    6187        18.92
12  2018-01    8037        19.07
13  2018-02    7518        18.59
14  2018-03    8017        20.86
15  2018-04    7674        20.76
16  2018-05    7595        19.91
17  2018-06    7010        22.24
18  2018-07    6960        22.96
19  2018-08    7135        20.57


**Nhận xét**: Phí ship dao động nhẹ quanh mức 17–23 R$ qua các tháng, tăng nhẹ vào giữa năm 2018 (tháng 6–7, ~22–23 R$) — không có biến động mùa vụ cực đoan, cho thấy yếu tố thời gian có ảnh hưởng yếu hơn nhiều so với khoảng cách/trọng lượng.

---
## Query 7 — Tỷ lệ phí ship trên giá sản phẩm, theo nhóm giá

**Câu hỏi phân tích**: Phí ship chiếm bao nhiêu phần trăm giá trị đơn hàng, và tỷ lệ này có khác nhau giữa sản phẩm rẻ và đắt không?
**Mục đích**: Đây là chỉ số quan trọng về mặt kinh doanh — phí ship cố định tương đối (phụ thuộc trọng lượng/khoảng cách) sẽ chiếm tỷ trọng khác nhau tùy giá trị sản phẩm.

In [37]:
print("=== 7. TỶ LỆ PHÍ SHIP / GIÁ SẢN PHẨM THEO NHÓM GIÁ ===")
print(sql("""
SELECT
    CASE
        WHEN price < 50  THEN '1. Giá rẻ (<50)'
        WHEN price < 150 THEN '2. Trung bình (50-150)'
        WHEN price < 400 THEN '3. Cao (150-400)'
        ELSE '4. Rất cao (>400)'
    END AS nhom_gia,
    COUNT(*) AS so_don,
    ROUND(AVG(freight_value), 2) AS phi_ship_TB,
    ROUND(AVG(price), 2) AS gia_TB,
    ROUND(AVG(freight_value * 1.0 / price) * 100, 1) AS ty_le_phi_tren_gia_pct
FROM freight_data
GROUP BY nhom_gia
ORDER BY nhom_gia;
"""))


=== 7. TỶ LỆ PHÍ SHIP / GIÁ SẢN PHẨM THEO NHÓM GIÁ ===
                 nhom_gia  so_don  phi_ship_TB  gia_TB  ty_le_phi_tren_gia_pct
0         1. Giá rẻ (<50)   38112        14.74   31.31                    57.0
1  2. Trung bình (50-150)   49355        19.54   92.54                    22.4
2        3. Cao (150-400)   17939        27.00  227.15                    12.4
3       4. Rất cao (>400)    4067        44.55  812.42                     6.2


**Nhận xét — đây là phát hiện đáng chú ý nhất trong toàn bộ phân tích SQL**: với sản phẩm giá rẻ (<50 R$), phí ship chiếm tới **57% giá trị sản phẩm**, trong khi với sản phẩm rất đắt (>400 R$), tỷ lệ này chỉ còn **6,2%**. Điều này có ý nghĩa thực tiễn quan trọng: phí vận chuyển là gánh nặng tương đối lớn hơn nhiều đối với các mặt hàng giá trị thấp, gợi ý rằng người bán mặt hàng giá rẻ cần cân nhắc chiến lược gộp đơn hoặc trợ giá vận chuyển.

---
## Query 8 — Top 10 tuyến đường (bang người bán → bang người mua) có phí ship cao nhất

**Câu hỏi phân tích**: Những tuyến vận chuyển cụ thể nào (từ bang nào đến bang nào) có chi phí cao nhất?
**Mục đích**: Phân tích chi tiết hơn Query 5 (vốn chỉ xét theo bang người bán) — xác định rõ cặp tuyến đường cụ thể, hữu ích cho việc tối ưu mạng lưới logistics (chỉ xét tuyến có ≥20 đơn).

In [38]:
print("=== 8. TOP 10 TUYẾN ĐƯỜNG (SELLER STATE -> CUSTOMER STATE) PHÍ SHIP CAO NHẤT ===")
print(sql("""
SELECT
    seller_state || ' -> ' || customer_state AS tuyen_duong,
    COUNT(*) AS so_don,
    ROUND(AVG(freight_value), 2) AS phi_ship_TB,
    ROUND(AVG(distance_km), 1) AS khoang_cach_TB
FROM freight_data
GROUP BY seller_state, customer_state
HAVING COUNT(*) >= 20
ORDER BY phi_ship_TB DESC
LIMIT 10;
"""))


=== 8. TOP 10 TUYẾN ĐƯỜNG (SELLER STATE -> CUSTOMER STATE) PHÍ SHIP CAO NHẤT ===
  tuyen_duong  so_don  phi_ship_TB  khoang_cach_TB
0    SC -> PB      20        72.47          2670.1
1    SC -> CE      22        68.48          2775.8
2    SC -> PA      21        59.94          2714.3
3    PR -> PI      30        59.29          2298.7
4    PR -> PA      58        52.95          2500.9
5    MG -> AL      38        52.55          1508.1
6    SC -> PE      39        51.13          2562.2
7    RS -> BA      49        50.99          2172.0
8    CE -> SP      24        50.83          2323.0
9    SC -> MA      22        50.43          2665.1


**Nhận xét**: Các tuyến đắt nhất đều xuất phát từ Santa Catarina (SC) và Paraná (PR) đến các bang vùng Đông Bắc/Bắc Brazil (PB, CE, PA, PI, MA) — khoảng cách trung bình trên 2.000km, xác nhận chi tiết hơn phát hiện ở Query 5.

---
## Query 9 — Phí ship theo số lượng sản phẩm trong đơn hàng

**Câu hỏi phân tích**: Đơn hàng mua nhiều sản phẩm cùng lúc có phí ship trung bình/sản phẩm thấp hơn không (hiệu ứng gộp đơn)?
**Mục đích**: Kiểm tra xem việc gộp nhiều sản phẩm vào 1 đơn có giúp tiết kiệm phí vận chuyển hay không — gợi ý hữu ích cho người bán.

In [39]:
print("=== 9. PHÍ SHIP THEO SỐ LƯỢNG SẢN PHẨM TRONG ĐƠN (order_n_items) ===")
print(sql("""
SELECT
    CASE WHEN order_n_items = 1 THEN '1 sản phẩm' ELSE '>=2 sản phẩm' END AS nhom,
    COUNT(*) AS so_don,
    ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM freight_data
GROUP BY nhom;
"""))


=== 9. PHÍ SHIP THEO SỐ LƯỢNG SẢN PHẨM TRONG ĐƠN (order_n_items) ===
           nhom  so_don  phi_ship_TB
0    1 sản phẩm   86289         20.4
1  >=2 sản phẩm   23184         18.6


**Nhận xét**: Đơn có từ 2 sản phẩm trở lên có phí ship trung bình/dòng thấp hơn một chút (18,6 R$ so với 20,4 R$) — gợi ý có hiệu ứng tiết kiệm nhẹ khi gộp đơn, dù không quá lớn.

---
## Query 10 — Kiểm tra chất lượng dữ liệu: tỷ lệ dòng bị suy diễn (imputed) theo bang người bán

**Câu hỏi phân tích**: Dữ liệu tọa độ/danh mục bị thiếu và phải suy diễn (impute) có phân bố đồng đều giữa các bang không, hay tập trung ở 1 số khu vực cụ thể?
**Mục đích**: Đây là bước kiểm tra chất lượng dữ liệu (data quality check) bổ sung — đảm bảo kết quả phân tích ở các bang nhỏ không bị sai lệch do tỷ lệ dữ liệu suy diễn quá cao (chỉ xét bang có ≥50 đơn).

In [40]:
print("=== 10. KIỂM TRA CHẤT LƯỢNG DỮ LIỆU - TỶ LỆ DÒNG BỊ IMPUTED THEO BANG ===")
print(sql("""
SELECT
    seller_state,
    COUNT(*) AS so_don,
    ROUND(AVG(geo_imputed) * 100, 1) AS pct_geo_imputed,
    ROUND(AVG(category_imputed) * 100, 1) AS pct_category_imputed
FROM freight_data
GROUP BY seller_state
HAVING COUNT(*) >= 50
ORDER BY pct_geo_imputed DESC
LIMIT 10;
"""))


=== 10. KIỂM TRA CHẤT LƯỢNG DỮ LIỆU - TỶ LỆ DÒNG BỊ IMPUTED THEO BANG ===
  seller_state  so_don  pct_geo_imputed  pct_category_imputed
0           DF     881              4.9                   1.1
1           CE      90              1.1                   0.0
2           MG    8579              1.0                   2.3
3           BA     623              0.8                   0.0
4           SP   78013              0.5                   1.4
5           MA     402              0.5                   0.0
6           GO     508              0.4                   1.8
7           RS    2162              0.3                   2.1
8           RJ    4646              0.3                   1.8
9           PR    8444              0.2                   0.7


**Nhận xét**: Tỷ lệ dữ liệu bị suy diễn (imputed) ở mọi bang đều rất thấp (dưới 5%), cao nhất là bang DF (4,9%) — xác nhận chất lượng dữ liệu đồng đều, không có bang nào bị sai lệch đáng kể do thiếu dữ liệu gốc.

In [41]:
conn.close()
print('\nĐã đóng kết nối SQLite.')



Đã đóng kết nối SQLite.
